# nba_api exploration

Not a general tour of the package - `nba_api` exposes
dozens of endpoints; this notebook stays narrow to what `src/` 
imports, so it relates to pipeline rather than unrelated exploration.

## 1. Static team metadata - `teams`

`nba_api.stats.static.teams` is not an API call at all - it's a list of 30
dicts shipped inside the package itself. No request, no rate limit, no
`sleep()`.

In [1]:
from nba_api.stats.static import teams

nba_teams = teams.get_teams()
print(f"Number of teams: {len(nba_teams)}")
nba_teams[0]

Number of teams: 30


{'id': 1610612737,
 'full_name': 'Atlanta Hawks',
 'abbreviation': 'ATL',
 'nickname': 'Hawks',
 'city': 'Atlanta',
 'state': 'Georgia',
 'year_founded': 1949}

## 2. Played games - `leaguegamefinder.LeagueGameFinder`

The main data source. One row per **team per game** — a single game shows up
twice, once for each side.

In [2]:
from nba_api.stats.endpoints import leaguegamefinder

gf = leaguegamefinder.LeagueGameFinder(
    season_nullable='2024-25',
    league_id_nullable='00',
    season_type_nullable='Regular Season'
)
df = gf.get_data_frames()[0]

print(df.shape)
print(df.head())

(2460, 28)
  SEASON_ID     TEAM_ID TEAM_ABBREVIATION          TEAM_NAME     GAME_ID  \
0     22024  1610612763               MEM  Memphis Grizzlies  0022401194   
1     22024  1610612742               DAL   Dallas Mavericks  0022401194   
2     22024  1610612759               SAS  San Antonio Spurs  0022401197   
3     22024  1610612766               CHA  Charlotte Hornets  0022401187   
4     22024  1610612756               PHX       Phoenix Suns  0022401200   

    GAME_DATE      MATCHUP WL  MIN  PTS  ...  FT_PCT  OREB  DREB  REB  AST  \
0  2025-04-13  MEM vs. DAL  W  239  132  ...   0.867    19    34   53   31   
1  2025-04-13    DAL @ MEM  L  240   97  ...   0.727    10    28   38   20   
2  2025-04-13  SAS vs. TOR  W  240  125  ...   0.875     9    41   50   22   
3  2025-04-13    CHA @ BOS  L  240   86  ...   0.611    18    33   51   23   
4  2025-04-13    PHX @ SAC  L  241   98  ...   0.500     9    28   37   28   

   STL  BLK  TOV  PF  PLUS_MINUS  
0   13    7   11  15        

Two rows per game is exactly why `clean_data.py` has a `split_home_away` /
`merge_home_away` step - the raw feed is per-team, and the pipeline turns it
into one row per game.

There's a second reason `clean_data.py` exists:
`MATCHUP` is occasionally corrupted - it doesn't start with the row's own
`TEAM_ABBREVIATION`. `fix_corrupted_matchup` repairs it by looking at the
other row for the same `GAME_ID`. Below: how common that actually is, in this
one season.

In [3]:
corrupted = df[df['TEAM_ABBREVIATION'] != df['MATCHUP'].str[:3]]
print(f"Corrupted MATCHUP rows: {len(corrupted)} / {len(df)}")
corrupted[['GAME_ID', 'TEAM_ABBREVIATION', 'MATCHUP']].head()

Corrupted MATCHUP rows: 5 / 2460


,GAME_ID,TEAM_ABBREVIATION,MATCHUP
1131,0022400633,SAS,IND @ SAS
1156,0022400621,IND,SAS @ IND
1710,0022401230,OKC,HOU @ OKC
1711,0022401229,MIL,ATL @ MIL
2282,0022400147,WAS,MIA @ WAS


## 3. Upcoming games - `scheduleleaguev2.ScheduleLeagueV2`

`LeagueGameFinder` above only ever returns games that have been **played**.
This is the only endpoint that returns games that
**haven't** - the source for every live prediction. It returns two
dataframes: `[0]` is the games themselves, `[1]` is a week-number/date
lookup.

In [4]:
from nba_api.stats.endpoints import scheduleleaguev2

s = scheduleleaguev2.ScheduleLeagueV2(season='2026-27', league_id='00')
dfs = s.get_data_frames()

print(f"{len(dfs)} dataframes returned")
print("games:", dfs[0].shape)
print("weeks:", dfs[1].shape)
games = dfs[0]
games[['gameId', 'gameDate', 'homeTeam_teamTricode', 'awayTeam_teamTricode']].head()

2 dataframes returned
games: (1274, 160)
weeks: (26, 6)


,gameId,gameDate,homeTeam_teamTricode,awayTeam_teamTricode
0,0012600009,10/03/2026 00:00:00,TOR,MIA
1,0012600066,10/04/2026 00:00:00,LAC,GSW
2,0012600067,10/04/2026 00:00:00,DEN,UTA
3,0012600004,10/05/2026 00:00:00,ATL,MEM
4,0012600022,10/05/2026 00:00:00,DET,PHX


- **`gameId` prefix** distinguishes game types - `001` preseason, `002`
  regular season, `004` playoffs, `005` play-in, `006` NBA Cup final. The
  model is trained on regular season only, so we keep `002`
  and drops the rest.
- **NBA Cup knockout games** are `002` (they count in the standings) but are
  published with blank team columns until the group stage decides who
  plays - we need to drop these too because there's nothing to predict
  about a game whose teams aren't known yet.

In [5]:
print(games['gameId'].str[:3].value_counts())

blank = games[games['homeTeam_teamTricode'].isna() | (games['homeTeam_teamTricode'] == '')]
print(f"\nGames with undecided participants (NBA Cup knockout): {len(blank)}")
blank[['gameId', 'gameDate', 'gameLabel']].head()

gameId
002    1206
001      67
006       1
Name: count, dtype: int64

Games with undecided participants (NBA Cup knockout): 7


,gameId,gameDate,gameLabel
392,0022601201,12/04/2026 00:00:00,Emirates NBA Cup
393,0022601202,12/04/2026 00:00:00,Emirates NBA Cup
394,0022601203,12/05/2026 00:00:00,Emirates NBA Cup
395,0022601204,12/05/2026 00:00:00,Emirates NBA Cup
396,0022601229,12/08/2026 00:00:00,Emirates NBA Cup


## How used in project

| call | returns | feeds |
| --- | --- | --- |
| `teams.get_teams()` | static, 30 teams | `src/fetch_teams.py` → `data/processed/teams.csv` |
| `LeagueGameFinder` | played games, 1 row/team/game | `src/fetch_data.py` → `data/raw/games.csv` |
| `ScheduleLeagueV2` | full season schedule | `src/fetch_schedule.py` → `data/raw/schedule.csv` |
